# 00 — Exploratory Data Analysis

Inspect both datasets before training:
- **PlantVillage** (lab images): folder-per-class structure, 38 classes
- **Plant Pathology 2021** (field images): CSV labels, multi-label format

Goals: count images per class, spot imbalance, check image sizes, visualise samples.

In [ ]:
import random
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT      = Path("..")
PV_COLOR  = ROOT / "data" / "plantvillage" / "plantvillage dataset" / "color"
PP_IMAGES = ROOT / "data" / "plantpathology" / "train_images"
PP_CSV    = ROOT / "data" / "plantpathology" / "train.csv"

print("PlantVillage color dir exists:", PV_COLOR.exists())
print("Plant Pathology images dir exists:", PP_IMAGES.exists())
print("Plant Pathology CSV exists:    ", PP_CSV.exists())


## 1. PlantVillage — class counts

In [ ]:
pv_classes = sorted([d.name for d in PV_COLOR.iterdir() if d.is_dir()])
pv_counts  = {cls: len(list((PV_COLOR / cls).glob("*.*"))) for cls in pv_classes}

pv_df = pd.DataFrame({"class": list(pv_counts.keys()), "count": list(pv_counts.values())})
pv_df = pv_df.sort_values("count", ascending=False).reset_index(drop=True)

print(f"Classes: {len(pv_df)}   |   Total images: {pv_df['count'].sum():,}")
print(f"Min class: {pv_df['count'].min()}  |  Max class: {pv_df['count'].max()}  |  Mean: {pv_df['count'].mean():.0f}")
pv_df.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
sns.barplot(data=pv_df, x="class", y="count", hue="class", legend=False, ax=ax, palette="viridis")
plt.setp(ax.get_xticklabels(), rotation=90, fontsize=8)
ax.set_title("PlantVillage - images per class (color subset)")
ax.set_xlabel("")
ax.set_ylabel("Image count")
plt.tight_layout()
plt.savefig(ROOT / "models" / "pv_class_dist.png", dpi=150)
plt.show()


## 2. Plant Pathology 2021 — label distribution

In [ ]:
pp_df = pd.read_csv(PP_CSV)
print(f"Rows: {len(pp_df)}   Columns: {list(pp_df.columns)}")
pp_df.head()

In [ ]:
all_labels = []
for label_str in pp_df["labels"]:
    all_labels.extend(str(label_str).split())

pp_label_counts = Counter(all_labels)
pp_label_df = pd.DataFrame(pp_label_counts.items(), columns=["label", "count"]).sort_values("count", ascending=False)

print(f"Unique labels: {len(pp_label_df)}   |   Total annotations: {sum(pp_label_counts.values()):,}")
print(f"\nLabel frequencies:\n{pp_label_df.to_string(index=False)}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=pp_label_df, x="label", y="count", hue="label", legend=False, ax=ax, palette="magma")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", fontsize=9)
ax.set_title("Plant Pathology 2021 - annotation frequency per label")
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig(ROOT / "models" / "pp_label_dist.png", dpi=150)
plt.show()


## 3. Image size distribution (PlantVillage sample)

In [ ]:
sample_paths = []
for cls in pv_classes:
    imgs = list((PV_COLOR / cls).glob("*.*"))
    sample_paths.extend(random.sample(imgs, min(4, len(imgs))))  # 4 per class = ~150 total

widths, heights = [], []
for p in tqdm(sample_paths, desc="Reading sizes"):
    try:
        w, h = Image.open(p).size
        widths.append(w)
        heights.append(h)
    except Exception:
        pass

print(f"Sampled {len(widths)} images")
print(f"Width  — min:{min(widths)} max:{max(widths)} mean:{np.mean(widths):.0f}")
print(f"Height — min:{min(heights)} max:{max(heights)} mean:{np.mean(heights):.0f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(widths, bins=30, color="steelblue")
axes[0].set_title("Width distribution")
axes[0].set_xlabel("px")
axes[1].hist(heights, bins=30, color="coral")
axes[1].set_title("Height distribution")
axes[1].set_xlabel("px")
plt.tight_layout()
plt.show()

## 4. Sample images — PlantVillage

In [ ]:
sample_classes = random.sample(pv_classes, 8)
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, cls in zip(axes.flat, sample_classes):
    imgs = list((PV_COLOR / cls).glob("*.*"))
    img = Image.open(random.choice(imgs))
    img.thumbnail((256, 256))  # resize in memory — faster than loading full res
    ax.imshow(np.array(img))
    ax.set_title(cls.replace("___", "\n"), fontsize=7)
    ax.axis("off")
plt.suptitle("PlantVillage - sample images (8 classes)", fontsize=11)
plt.tight_layout()
plt.show()


## 5. Sample images — Plant Pathology 2021

In [ ]:
actual_files = {p.name for p in PP_IMAGES.glob("*.*")}
pp_df_valid  = pp_df[pp_df["image"].astype(str).isin(actual_files)].reset_index(drop=True)
print(f"Total CSV rows : {len(pp_df):,}")
print(f"Images on disk : {len(actual_files):,}  ({len(pp_df) - len(pp_df_valid):,} missing — re-extract zip if > 0)")

sample_rows = pp_df_valid.sample(min(8, len(pp_df_valid)), random_state=SEED)
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, (_, row) in zip(axes.flat, sample_rows.iterrows()):
    img = Image.open(PP_IMAGES / row["image"])
    img.thumbnail((256, 256))
    ax.imshow(np.array(img))
    ax.set_title(str(row["labels"]), fontsize=7)
    ax.axis("off")
plt.suptitle("Plant Pathology 2021 - sample field images", fontsize=11)
plt.tight_layout()
plt.show()


## 6. Summary

In [ ]:
print("=" * 55)
print("DATASET SUMMARY")
print("=" * 55)
print(f"\nPlantVillage (color, lab images)")
print(f"  Classes : {len(pv_df)}")
print(f"  Images  : {pv_df['count'].sum():,}")
print(f"  Imbalance ratio (max/min): {pv_df['count'].max() / pv_df['count'].min():.1f}x")

print(f"\nPlant Pathology 2021 (field images, multi-label)")
print(f"  Images  : {len(pp_df):,}")
print(f"  Unique labels : {len(pp_label_df)}")
print(f"  Most common   : {pp_label_df.iloc[0]['label']} ({pp_label_df.iloc[0]['count']:,})")
print(f"  Least common  : {pp_label_df.iloc[-1]['label']} ({pp_label_df.iloc[-1]['count']:,})")
print("=" * 55)